In [1]:
import os

expected = ["agents", "tools", "graph", "state", "tests", "data", "docs"]
for folder in expected:
    exists = os.path.isdir(folder)
    print(f"{'✅' if exists else '❌'} {folder}/")

✅ agents/
✅ tools/
✅ graph/
✅ state/
✅ tests/
✅ data/
✅ docs/


In [2]:
%%writefile state/schema.py
from typing import TypedDict, Annotated, List, Dict, Optional
from operator import add


class ResearchState(TypedDict):
    """Shared state that flows through every node in the research graph."""
    query: str
    plan: List[str]
    needs_more_research: bool

    # fields multiple parallel agents may write to — append, don't overwrite
    search_results: Annotated[List[Dict], add]
    extracted_facts: Annotated[List[Dict], add]
    fact_check_results: Annotated[List[Dict], add]
    summaries: Annotated[List[Dict], add]

    final_report: Optional[str]

    # observability — which model handled which agent (LangSmith covers detailed tracing;
    # this stays for report-time attribution, e.g. citing which sources fed the final report)
    model_usage_log: Annotated[List[Dict], add]

Writing state/schema.py


In [3]:
%%writefile tools/llm_clients.py
import os
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
from tavily import TavilyClient

load_dotenv()

GROQ_KEY = os.getenv("GROQ_API_KEY")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")

assert GROQ_KEY, "Missing GROQ_API_KEY"
assert GEMINI_KEY, "Missing GEMINI_API_KEY"
assert TAVILY_KEY, "Missing TAVILY_API_KEY"

reasoning_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)


gemini_client = genai.Client(api_key=GEMINI_KEY)
tavily_client = TavilyClient(api_key=TAVILY_KEY)


def research_with_fallback(query: str) -> dict:
    """Tries Gemini (grounded search + citations) first. Falls back to Tavily
    automatically on any Gemini failure, so the pipeline never fully blocks
    on one provider's outage."""
    try:
        resp = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=query,
            config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
        )
        sources = []
        if resp.candidates[0].grounding_metadata:
            sources = [
                {"title": c.web.title, "url": c.web.uri}
                for c in resp.candidates[0].grounding_metadata.grounding_chunks
            ]
        return {"answer": resp.text, "sources": sources, "provider": "gemini"}
    except Exception as e:
        print(f"Gemini unavailable ({type(e).__name__}), falling back to Tavily...")
        results = tavily_client.search(query=query, max_results=3)
        sources = [{"title": r["title"], "url": r["url"]} for r in results["results"]]
        answer = " ".join(r.get("content", "")[:300] for r in results["results"])
        return {"answer": answer, "sources": sources, "provider": "tavily"}

Writing tools/llm_clients.py


In [4]:
%%writefile tools/search_tool.py
import time
from langchain_core.tools import tool
from tools.llm_clients import tavily_client


@tool
def web_search(query: str) -> str:
    """Search the web for current information on a topic. Use this when you need
    up-to-date facts, news, or information that might not be in your training data."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            results = tavily_client.search(query=query, max_results=3)
            formatted = "\n\n".join(
                f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content'][:300]}"
                for r in results["results"]
            )
            return formatted
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f" Search failed ({e.__class__.__name__}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                return f"Search failed after {max_retries} attempts: {e}"

Writing tools/search_tool.py


In [5]:
%%writefile tools/logger.py
import logging
import os

os.makedirs("data/logs", exist_ok=True)

logger = logging.getLogger("research_system")
logger.setLevel(logging.INFO)

if not logger.handlers:
    file_handler = logging.FileHandler("data/logs/system.log")
    console_handler = logging.StreamHandler()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    )
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)


def log_agent_event(agent_name: str, event: str, **kwargs):
    """Structured log line: agent name, what happened, and any extra context."""
    extra = " ".join(f"{k}={v}" for k, v in kwargs.items())
    logger.info(f"[{agent_name}] {event} {extra}".strip())

Writing tools/logger.py


In [6]:
import sys
sys.path.append(".")  

from state.schema import ResearchState
from tools.llm_clients import reasoning_llm, research_with_fallback
from tools.search_tool import web_search
from tools.logger import log_agent_event

print(" ResearchState imported:", list(ResearchState.__annotations__.keys()))

log_agent_event("test_agent", "smoke_test_passed", detail="Day 6 module refactor")

response = reasoning_llm.invoke("Reply with exactly: modules working correctly.")
print(" reasoning_llm:", response.content)

print("web_search tool name:", web_search.name)

2026-09-27 13:14:37,842 | INFO | research_system | [test_agent] smoke_test_passed detail=Day 6 module refactor


 ResearchState imported: ['query', 'plan', 'needs_more_research', 'search_results', 'extracted_facts', 'fact_check_results', 'summaries', 'final_report', 'model_usage_log']
 reasoning_llm: modules working correctly.
web_search tool name: web_search


In [7]:
%%writefile requirements.txt
langgraph
langchain
langchain-groq
langchain-community
pydantic
python-dotenv
tavily-python
google-genai
langsmith

Writing requirements.txt


In [8]:
%%writefile .gitignore
.env
__pycache__/
*.pyc
venv/
.ipynb_checkpoints/
data/logs/
*.log

Overwriting .gitignore


In [ ]:
%%writefile README.md
# Multi-Agent Research & Intelligence System

A LangGraph-based multi-agent system with a Supervisor/Planner Agent coordinating
specialized agents (Web Search, Data Extraction, Fact-Checking, Summarization,
Report Generation) to research a topic and produce a cited, structured report.

## Architecture
See `docs/architecture.md` for the full diagram and design decisions.

- **Supervisor/Planner, Fact-Checker, Summarizer, Report Generator** — Groq (qwen/qwen3.8-27b)
- **Web Search Agent** — Gemini (grounded search) with automatic Tavily fallback

## Setup
1. `python -m venv venv && source venv/bin/activate` (or `venv\\Scripts\\activate` on Windows)
2. `pip install -r requirements.txt`
3. Create a `.env` file with `GROQ_API_KEY`, `GEMINI_API_KEY`, `TAVILY_API_KEY`, and optionally `LANGSMITH_API_KEY` + `LANGSMITH_TRACING=true`

## Project structure
- `agents/` — individual agent logic
- `tools/` — reusable tools and LLM client setup
- `graph/` — LangGraph orchestration
- `state/` — shared state schema
- `tests/` — unit and integration tests
- `docs/` — architecture notes and diagrams